# Using `matcher.py`

This notebook shows how to import and use `AdaptiveTemplateMatching` from the packaged module in `src/adaptive_template_matching/matcher.py`.

Before running the examples, install the package from the repository root:

```bash
pip install -e .
```

## 1. Import the matcher

The package exposes `AdaptiveTemplateMatching` at the top level.

In [ ]:
from adaptive_template_matching import AdaptiveTemplateMatching
import numpy as np

## 2. Create a matcher instance

You can use the defaults or customize the template geometry.

In [ ]:
matcher = AdaptiveTemplateMatching(
    cp_inds=[145, 195],
    template_angls=[80, 88],
    template_len=200,
    baseline=0.0,
    template_scaler=0.12,
    reflect=False,
)

matcher

## 3. Generate or load signal data

Replace the synthetic example below with your real gait signal array when needed.

In [ ]:
x = np.linspace(0, 12 * np.pi, 4000)
signal = 0.03 * np.sin(x) + 0.01 * np.random.randn(len(x))

# Add a few template-like events to the signal.
template = matcher.get_template_shape()
for start in [500, 1400, 2500, 3300]:
    stop = start + len(template)
    if stop <= len(signal):
        signal[start:stop] += template

## 4. Run a scan

Tune the thresholds to match your signal scaling and data quality.

In [ ]:
indices, scores = matcher.scan_matches(
    data=signal,
    threshold=0.6,
    amp_max=1.0,
    rel_err=0.5,
    flat_err_thresh=0.2,
    pos_shift=20,
    sad_thresh=0.0,
    min_std=1e-3,
    enforce_low_amp=False,
    show_debug=False,
)

indices, scores[:10] if len(scores) else scores

## 5. Update the template

If you want the template to adapt to accepted segments, call `update_template`.

In [ ]:
matcher.update_template(signal, np.asarray(indices), show_debug=False)
matcher.get_template_shape()[:10]

## 6. Run dataset helpers

For multi-signal workflows, pass a dictionary of dataset names to signal arrays.

In [ ]:
data_dict = {
    "trial_1": signal,
    "trial_2": signal + 0.002 * np.random.randn(len(signal)),
}

results = matcher.cold_start_run_dataset(
    data_dict=data_dict,
    amp_max=1.0,
    thr=0.6,
    rel_err=0.5,
    passes=1,
    pos_shift=20,
    sad_thresh=0.0,
    min_std=1e-3,
    enforce_low_amp=False,
    show_debug=False,
    debug_verbose=False,
)

results

## Notes

- `scan_matches` is the lowest-level matching call.
- `final_template_match_and_plot` is useful when you want summary output and optional plots.
- `cold_start_run_dataset`, `warm_start_run_dataset`, and `all_data_run_dataset` are convenient for multi-dataset adaptation workflows.